# Baseball

"I think the discovery of new analytics is never ending. I think there will always be things to research, always new discoveries that we make in baseball alone, let alone all the other sports" -- John Dewan, Founder, STATS Inc.

"What we assume that the teams should know, but never seem to get, is that you're paying \[players\] for the future, not the past." -- Wharton Statistics Professor Abraham Wyner

"Baseball is 90% mental. The other half is physical." -- Yogi Berra

## Python Packages

We will use the following Python packages in this chapter. Some we have used in previous chapters.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

We will also introduce these packages which we have not yet used. If you do not have them installed yet on your computer, you can install them using `pip install <package-name>`.

In [ ]:
from mlb_utils import batting_stats_mlb, team_batting_mlb, team_pitching_mlb
from pybaseball import playerid_lookup, statcast_batter, statcast_pitcher
from pybaseball import cache
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

# Enable caching to speed up repeated data requests
cache.enable()

In [ ]:
#| echo: false
#| eval: true

# Set default plot style
sns.set_theme(style="whitegrid")

## Introduction

Major League Baseball (MLB), as Michael Lewis subtitled in his 2003 book *Moneyball*, is an unfair game. Despite the efforts of other professional sports leagues to level the ability to acquire players, such as through salary caps, baseball has never created this hard limit. For example, in 2023 the payroll of the highest spending team, the New York Mets at approximately \$343 million, was almost five times as high as the team with the lowest payroll, the Oakland Athletics at approximately \$62 million. In fact, the six highest payrolls in 2023 were from the two teams from New York, the Los Angeles Dodgers, and teams from Dallas, Philadelphia and San Diego, cities that rank 1, 2, 9, 6, and 8 by population in the US. These teams stand to make much more money from ticket sales, media rights, merchandise sales, and other ancillary income than teams in places like Pittsburgh, Baltimore, and Oakland which occupy the last three spots in payroll. Once free agency took place in the mid-1970s and player salaries climbed exponentially, teams in these larger markets could afford to pay players much higher salaries than those in the smaller markets.

Moreover, salaries of players in MLB are fully guaranteed. If a player signs a 10-year contract and his performance diminishes after 2 years, the team is obligated to honor that contract throughout its lifetime, even if the player is cut from the team. Making a mistake with a player whose performance declines can be an anchor around that team for years. Famously, Bobby Bonilla receives a \$1 million+ payment from the New York Mets every July 1st through 2031, even though he hasn't played for the team since 2000 ^[In December 2023 Shohei Ohtani signed a 10 year, $700 million contract with the Los Angeles Dodgers, almost doubling the largest previous contract in MLB history. This contract will have $680 million deferred, payable $68 million annually on July 1 from 2034 to 2043. July 1 will certainly then become known as Shohei Ohtani Day.]. If owners of teams in places like Oakland expected to compete with those in the massive media markets, they would have to rely on something besides their checkbooks.

This is where analytics came into play. Led by newer thinking in Oakland of people like GM Billy Beane and his assistant GM Paul DePodesta, they viewed the game differently and evaluated players on how they could make contributions to the team by creating runs or creating outs. They looked for players that were otherwise overlooked by conventional baseball personnel because they didn't pass the "eye test" or look like what was presumed to be a professional baseball player. They used this to great advantage in the early part of the 21st century, until other teams caught up. There continue to be innovations on this strategy, as you will read later in this chapter with the Houston Astros, and there are always new statistics to be measured as teams look for an edge.

In [ ]:
#| fig-cap: "Billy Beane and Paul DePodesta."
#| fig-subcap:
#|   - ""
#| label: fig-beane-dep
#| echo: false
#| layout-ncol: 2

from IPython.display import display, Image
import matplotlib.image as mpimg

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
img1 = mpimg.imread('fig/beane.jpeg')
img2 = mpimg.imread('fig/depodesta.jpeg')
axes[0].imshow(img1)
axes[0].axis('off')
axes[1].imshow(img2)
axes[1].axis('off')
plt.tight_layout()
plt.show()

So why baseball and analytics? Baseball players, coaches, and administrators need to be able to evaluate players for performance and game situations for the optimal strategy to score the most runs in 9 innings and win the game. In the first four chapters of this book, we have introduced you to sports analytics, a programming language and methodology for working with sports data, and how to acquire data to analyze from different sources. Over the next several chapters we will look at individual sports, not just baseball, and how to wrangle and analyze data related to those sports for advanced analysis. The first time we use a technique we will explain more about it, but the subsequent times that we use it we will assume you have seen it before. If you have jumped ahead to a particular chapter and do not understand a particular function or technique that is being used, refer back to previous chapters (and the index) to see where you might find out about that technique.

## If You Don't Know Anything about Baseball

Baseball, often known as the American Pastime, is a game played between two teams of nine players where the object is to score more runs than your opponents. Legend has it that the game was invented by Abner Doubleday in the 1830s, but is almost certainly derived from the English game of cricket and a more closely related game called rounders. It is also unique amongst sports in that the defense is the team that actually touches the ball, while the offense attempts to hit the ball where the defense cannot catch it or field it.

![Baseball field](fig/baseball-field.png){#fig-baseball-field}

The defense throws (pitches) the ball towards an offensive player, who then attempts to hit the ball with a bat. The nine defensive players then attempt to catch the ball before it touches the ground or to touch the player with the ball as the player runs toward first base, and then second base, and then third base, before arriving back where he started at home plate. If he reaches home plate, the offense scores one point (called a run); if he does not, it is called an out. When the defense records three outs, the teams change sides and the defense goes on offense and vice versa. An inning consists of each team taking a turn on offense and defense. The team with the most runs at the end of nine innings is the winner.

Obviously, this is an oversimplification of the game, but it gives you the basics of the game. A more complete explanation of the game is available at [https://www.wikihow.com/Play-Baseball](https://www.wikihow.com/Play-Baseball).

## Sabermetrics

More than a century's worth of analysis in baseball changed with the insights of Bill James and his publication of the Bill James Baseball Abstract in the late 1970s. In lieu of the traditional metrics used to evaluate players like batting average, runs, and runs batted in, or for pitchers wins, earned run average and strikeouts, James stressed new metrics like Runs Created, Pythagorean Win Percentage, and Game Score. He termed his approach to examining baseball players and teams *sabermetrics*, a nod to the Society for American Baseball Research. While many baseball purists dismissed James' work as a sideshow to the real analysis of baseball players by "baseball people", today baseball teams employ teams of analytics professionals to use mathematics, predictive analytics, and optimization to assist them in fielding the team with the best chance to win and in making in-game decisions which lead to the best chances to win.

![Bill James](fig/james.jpeg){#fig-james-head width=40%}

Many people know nothing more about sports analytics than what they may have gleaned while watching the 2011 movie *Moneyball*. In that movie, Bill James was referenced with derision in the scene where the scouts assemble to discuss which players the Oakland A's should sign for the 2002 season. In the time since that season chronicled in the movie, the analysts have certainly not replaced the scouts in the evaluation of players, but every team has and uses analysts and far more complex analytics.

This chapter shows you how to use some of those metrics that have been popularized in the last two decades, and perhaps even create some of your own!

## Basic Baseball Metrics

### Historical Metrics {.unnumbered}

For the first 100+ years of baseball, baseball players were measured largely by a few basic offensive and defensive metrics. However, with the advent of computers and Sabermetrics and analytics, more advanced metrics were created.

| **Offensive**                 | **Defensive**                            |
|-------------------------------|------------------------------------------|
| Batting Average: Hits/At bats | Wins: The pitcher who was pitching when the offense took the lead for good |
| Home Runs                     | Innings Pitched |
| Runs Batted In (RBI)          | Earned Run Average: The number of earned runs per 9 innings that a pitcher allowed the offense to score. |
| Walks                         | Strikeouts |
| On Base Percentage (OBP)      | WHIP: (Walks + Hits)/(Innings Pitched) |
| Runs Scored                   | Saves: A pitcher who came into the game when his team led by 3 or less runs and pitched effectively for at least one inning and his team wins the game. |
| Slugging Pct (SLG): TB/AB     | Fielding Percentage: percentage of time that a player handles the ball in the field and does not make an error. |

: Early metrics used in baseball analytics. {#tbl-early-baseball}

### Runs Created {.unnumbered}

*Runs Created* ($RC$), one of the original Bill James' metrics, is defined as
$$\textrm{runs created} = \frac{(hits + BB + HBP)\times TB}{AB + BB + HBP}.$$
(BB = walks; HBP = Hit By Pitch; AB = At Bats; TB = Total Bases.)
Runs Created is an estimate of how many runs should have been scored by a player or a team. This can be important when evaluating a player, or a team, for consideration for the future. For example, if we are considering adding a player to our team, one metric we might use to evaluate that player is to see how many runs he might be responsible for creating for our team and compare that to other players we might want to add to our team.

How can we utilize the pybaseball package to compute this metric? Let's get team batting data since 1970 using the `team_batting()` function.

In [ ]:
# Get team batting data for multiple years (1970-2025)
years_data = []
for year in range(1970, 2026):
    try:
        years_data.append(team_batting_mlb(year))
    except:
        pass  # Skip years with no data

team_df = pd.concat(years_data, ignore_index=True)
print(f"Total rows: {len(team_df)}")
team_df.head()

One thing to note is that the pybaseball `team_batting()` function provides many statistics directly. We can calculate Runs Created using the available columns. Let's calculate singles (1B) since it's not always explicitly provided, and then compute Runs Created.

In [ ]:
# Calculate singles and total bases
rc_df = team_df.copy()

# Calculate singles: H - 2B - 3B - HR
rc_df['X1B'] = rc_df['H'] - rc_df['2B'] - rc_df['3B'] - rc_df['HR']

# Calculate Total Bases
rc_df['TB'] = rc_df['X1B'] + 2*rc_df['2B'] + 3*rc_df['3B'] + 4*rc_df['HR']

# Handle missing HBP values (set to 0 if missing)
rc_df['HBP'] = rc_df['HBP'].fillna(0)

# Calculate Runs Created
rc_df['RC'] = (rc_df['H'] + rc_df['BB'] + rc_df['HBP']) * rc_df['TB'] / (rc_df['AB'] + rc_df['BB'] + rc_df['HBP'])

rc_df[['Team', 'yearID', 'R', 'H', 'HR', 'BB', 'TB', 'RC']].head()

In [ ]:
#| fig-cap: Calculated runs created (x) versus actual runs for all teams since 1970.
#| label: fig-runs-created-runs

sns.regplot(data=rc_df, x='RC', y='R',
            scatter_kws={'alpha': 0.15},
            line_kws={'color': 'black'})
plt.xlabel('Runs Created')
plt.ylabel('Actual Runs')
plt.title('Runs Created vs Actual Runs (Teams since 1970)')
plt.show()

@fig-runs-created-runs suggests that we could probably reliably predict a team's runs based on its runs created value. In other words, we can calculate a linear regression model between R and RC. If you don't know what linear regression is or how to calculate it, be sure to see a textbook such as *Applied Linear Regression Models* by Kutner, Nachsheim, and Neter for a primer. In this case, we will use the `statsmodels` package to create a regression model with runs (R) as the dependent variable and runs created (RC) as the independent variable. In familiar statistical terms, this model is written as
$$
R = \beta_0 + \beta_1 RC + \varepsilon.
$$

In [ ]:
# Fit a linear regression model
rc_r_lm = smf.ols('R ~ RC', data=rc_df).fit()
print(rc_r_lm.summary())

The resulting estimated regression equation is approximately:
$$
\hat{R} = 36.63 + 0.93 \cdot RC
$$
where $\hat{R}$ is referred to as "R-hat", or the estimated value of $R$.

We can use this equation to predict a team's average runs for a given value of runs created. Note that the predicted values for each observation can be obtained using the `predict()` method. We illustrate this concept by showing runs, runs created, and the predicted value of runs for a recent Colorado Rockies season.

In [ ]:
# Add predicted values to the dataframe
rc_df['R_hat'] = rc_r_lm.predict(rc_df)

# Filter for a specific team and year
rc_df[rc_df['Team'] == 'COL'][['Team', 'yearID', 'R', 'RC', 'R_hat']].tail(3)

The difference between the observed value of runs and the predicted value is referred to as the *residual* error. We will explore linear regression in more detail in the next section of this textbook.

Next we can check how well `RC` correlates with other team-based metrics. We will use pandas to create a correlation matrix between runs, runs created, at bats, hits, home runs, strikeouts, and stolen bases.

In [ ]:
#| fig-cap: Correlation matrix for runs, runs created, at bats, hits, home runs, strike outs, and stolen bases.
#| label: fig-corr-plot

# Select columns for correlation matrix
cols = ['R', 'RC', 'AB', 'H', 'HR', 'SO', 'SB']
cor_mat = rc_df[cols].corr()

# Plot correlation matrix
sns.heatmap(cor_mat, annot=True, cmap='RdYlBu_r', center=0,
            square=True, fmt='.2f')
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

We see strong correlations between RC and other forms of offensive production. What may further pique our interest, however, is that there is a modest positive correlation between the number of strikeouts and RC. Could it be that the team that swings for the fences is more likely to score enough runs that additional strikeouts can be tolerated? Or is this just a function of having more at bats? These are problems you might consider analyzing.

## Linear Regression

Rather than just looking at the correlation between runs and runs created, consider predicting runs scored in a season (per player) based on home runs using a simple linear regression model (see @fig-baseball-hr-vs-r). To be consistent with other examples within this text, we will:

* exclude anyone with less than 100 ABs so as to not have our model unduly influenced by non-full time position players;
* visually examine the relationship between R and HR by creating a scatterplot; and
* using linear regression to create a line of best fit.

This is similar to adding a trendline to a scatter plot in Microsoft Excel if you are familiar with that process.

In [ ]:
# Get individual batting data for recent years
batting_years = []
for year in range(2000, 2026):
    try:
        batting_years.append(batting_stats_mlb(year))
    except:
        pass

df_batting = pd.concat(batting_years, ignore_index=True)
df_batting = df_batting[df_batting["AB"] >= 100].copy()
print(f"Total player-seasons: {len(df_batting)}")

In [ ]:
#| label: fig-baseball-hr-vs-r
#| fig-cap: "Home runs (x) versus runs scored (y) by player since 2000."

sns.regplot(data=df_batting, x='HR', y='R',
            scatter_kws={'alpha': 0.15},
            line_kws={'color': 'black'})
plt.xlabel('Home Runs')
plt.ylabel('Runs')
plt.title('Home Runs vs Runs Scored (Players since 2000)')
plt.show()

In order to see the actual equation, we can run the actual linear regression model.

In [ ]:
lm_batting_1 = smf.ols('R ~ HR', data=df_batting).fit()
print(lm_batting_1.summary())

So how do we interpret the parameters of this model? When a player has zero home runs, the average number of runs scored by that player is approximately the y-intercept. Similarly, we expect that for each additional home run that a player hits, the average number of runs scored by that player will increase by approximately 2 runs.

We know that any point estimate, or prediction, from an estimated regression model has error. Therefore, we may wish instead to communicate a confidence interval for the parameters of our models using the `conf_int()` method. This lets us state with a given level of confidence that our parameter is between a lower bound and upper bound.

In [ ]:
# Get confidence intervals (99% level)
print(lm_batting_1.conf_int(alpha=0.01))

Suppose we want to store the results of our predicted values from our model with the actual data. You can calculate the predictions and append them directly onto your original data frame as a new column.

In [ ]:
df_batting_new = df_batting.copy()
df_batting_new['pred_R'] = lm_batting_1.predict(df_batting_new)
df_batting_new[['Name', 'Season', 'HR', 'R', 'pred_R']].head()

Alternatively, we can predict new observations by creating a data object that we pass to the `predict()` method. This allows us to make predictions using an existing model at potentially different values of our independent variables.

In [ ]:
# Predict for players with 30 and 15 home runs
new_data = pd.DataFrame({'HR': [30, 15]})
predictions = lm_batting_1.get_prediction(new_data)
pred_summary = predictions.summary_frame(alpha=0.05)
print(pred_summary[['mean', 'mean_ci_lower', 'mean_ci_upper']])

The `mean` column shows the predicted values of average runs scored at 30 and 15 home runs, and `mean_ci_lower` and `mean_ci_upper` are the lower and upper 95% confidence limits for those predictions.

## Player Comparisons using Runs Above Average

Earlier we discussed how we might use the Runs Created metric to compare and evaluate players that we might want to add to our team. In this section, we will look at how much better the "average" team would be if we could add a specific player. For example, we will look at replacing an average player with Aaron Judge in 2025. Judge, the New York Yankees slugger, has been one of the most dominant power hitters in baseball in recent years.

Suppose we decided that a model for predicting runs is based on singles, doubles, triples, home runs, and walks plus hit by pitches. Let's create a dataframe that:

* selects all the team-level data from 2010 to present;
* calculates and stores the number of singles in another column; and
* calculates and stores the sum of walks and hit by pitch into yet another column.

In [ ]:
# Get team data from 2010 to present
team_years = []
for year in range(2010, 2026):
    try:
        team_years.append(team_batting_mlb(year))
    except:
        pass

df_teams = pd.concat(team_years, ignore_index=True)

# Calculate singles and BB_HBP
df_teams["X1B"] = (df_teams["H"] - df_teams["2B"]
                   - df_teams["3B"] - df_teams["HR"])
df_teams["HBP"] = df_teams["HBP"].fillna(0)
df_teams["BB_HBP"] = df_teams["BB"] + df_teams["HBP"]

# Select relevant columns
df_teams_model = df_teams[
    ["yearID", "R", "X1B", "2B", "3B", "HR", "BB_HBP"]].copy()
df_teams_model.columns = ["yearID", "R", "X1B", "X2B", "X3B", "HR", "BB_HBP"]
df_teams_model.head()

### Multiple Regression {.unnumbered}

So far we have only looked at univariate regression - where we try to predict the value of a dependent variable from a single independent variable. These are perhaps more easily understood and definitely more easily visualized in two-dimensional space, but limit our power to look at multiple independent variables at the same time. Multiple regression allows us to look at many variables simultaneously in developing a model that predicts the value of the dependent variable.

The model that we are fitting is:
$$
R = \beta_0 + \beta_1 X1B + \beta_2 X2B + \beta_3 X3B + \beta_4 HR + \beta_5 (BB\_HBP) + \varepsilon.
$$

In [ ]:
# Fit multiple regression model
lm_runs = smf.ols('R ~ X1B + X2B + X3B + HR + BB_HBP', data=df_teams_model).fit()
print(lm_runs.summary())

### Model Assumptions {.unnumbered}

Note that when doing inference (e.g., testing the significance of a parameter value) in regression, several theoretical properties related to the model are assumed. These include:

* The true relationship between the independent and dependent variables is linear
* The residuals (errors) are independent from each other
* The residuals are normally distributed
* The residuals have a constant level of variance (are homoscedastic)

It is therefore important to assess the validity of these assumptions. We can create residual plots and a normal probability plot to check these assumptions.

In [ ]:
#| fig-cap: "Diagnostic plots for the multiple regression model."
#| label: fig-diagnostic-plots

fig, axes = plt.subplots(2, 2, figsize=(10, 8))

# Residuals vs Fitted
axes[0, 0].scatter(lm_runs.fittedvalues, lm_runs.resid, alpha=0.5)
axes[0, 0].axhline(y=0, color='red', linestyle='--')
axes[0, 0].set_xlabel('Fitted Values')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Residuals vs Fitted')

# Q-Q plot
stats.probplot(lm_runs.resid, dist="norm", plot=axes[0, 1])
axes[0, 1].set_title('Normal Q-Q Plot')

# Scale-Location plot
axes[1, 0].scatter(lm_runs.fittedvalues, np.sqrt(np.abs(lm_runs.resid)), alpha=0.5)
axes[1, 0].set_xlabel('Fitted Values')
axes[1, 0].set_ylabel('Sqrt(|Residuals|)')
axes[1, 0].set_title('Scale-Location')

# Residuals histogram
axes[1, 1].hist(lm_runs.resid, bins=30, edgecolor='black')
axes[1, 1].set_xlabel('Residuals')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Residuals Distribution')

plt.tight_layout()
plt.show()

Looking at the plots above, there does not appear to be any noticeable violations of the model assumptions.

## Runs Above Average: Aaron Judge (2025)

In this example, we will show how a specific player, in this case Aaron Judge in 2025, would impact an "average" team. The comparison of Judge to an "average player" is based on two calculations:

1. compute the average team prediction of runs using the regression model, and
2. adjust this average for Judge's impact by calculating a scaling factor.

We will need to move from looking at team-based metrics to the player-specific metrics. We will utilize the pybaseball package to get individual batting statistics.

First, let's get Aaron Judge's 2025 batting statistics using pybaseball.

In [ ]:
# Get batting stats for 2025
batting_2025 = batting_stats_mlb(2025)

# Find Aaron Judge
judge = batting_2025[batting_2025['Name'].str.contains('Aaron Judge', case=False)].copy()

# Calculate additional metrics
judge['X1B'] = judge['H'] - judge['2B'] - judge['3B'] - judge['HR']
judge['BB_HBP'] = judge['BB'] + judge['IBB'] + judge['HBP']
# Calculate Outs (approximate)
judge['Outs'] = 0.982 * judge['AB'] - judge['H'] + judge['GDP'] + judge['SF'] + judge['SH'] + judge['CS']

judge[['Name', 'Team', 'R', 'X1B', '2B', '3B', 'HR', 'BB_HBP', 'Outs']]

The average team in this context is defined by the average values across all statistical categories. We then find the expected runs based on the average statistics.

In [ ]:
# Calculate average team for 2025
avg_team = df_teams_model[df_teams_model['yearID'] == 2025].mean()
avg_team_df = pd.DataFrame([avg_team])

# Predict runs for average team
avg_team_runs = lm_runs.predict(avg_team_df)[0]
print(f"Average team predicted runs: {avg_team_runs:.1f}")

Next, we need the average number of outs per team during the 2025 season. We'll estimate this from the team data.

In [ ]:
# Estimate average outs per team (approximately AB - H + other outs)
# Using team batting data for 2025
team_2025 = df_teams[df_teams['yearID'] == 2025].copy()

# Approximate team outs (this is simplified)
avg_outs = (team_2025['AB'].mean() - team_2025['H'].mean()) * 3  # rough approximation for IPouts
print(f"Approximate average team outs: {avg_outs:.0f}")

The player-specific adjustment is based on how many outs the player (Judge) makes over the course of the season. The scaling factor for Judge is:
$$
\textrm{scaling factor} = 1 - \frac{\textrm{Judge's Outs}}{\textrm{Average Team Outs}}
$$

In [ ]:
# Calculate scaling factor
judge_outs = judge['Outs'].values[0]
judge_scale = (avg_outs - judge_outs) / avg_outs
print(f"Judge's outs: {judge_outs:.0f}")
print(f"Scaling factor: {judge_scale:.3f}")

Now we scale the average team according to the outs made by Judge and add his statistics.

In [ ]:
# Create average team with Judge
judge_stats = judge[['R', 'X1B', 'BB_HBP']].copy()
judge_stats.columns = ['R', 'X1B', 'BB_HBP']
judge_stats['X2B'] = judge['2B'].values[0]
judge_stats['X3B'] = judge['3B'].values[0]
judge_stats['HR'] = judge['HR'].values[0]

# Scale average team and add Judge
avg_team_scaled = avg_team[['R', 'X1B', 'X2B', 'X3B', 'HR', 'BB_HBP']] * judge_scale
avg_team_w_judge = pd.DataFrame([avg_team_scaled.values + judge_stats[['R', 'X1B', 'X2B', 'X3B', 'HR', 'BB_HBP']].values[0]],
                                 columns=['R', 'X1B', 'X2B', 'X3B', 'HR', 'BB_HBP'])

# Predict runs with Judge
runs_w_judge = lm_runs.predict(avg_team_w_judge)[0]

# Calculate Runs Above Average
raa_judge = runs_w_judge - avg_team_runs
print(f"Aaron Judge's Runs Above Average (2025): {raa_judge:.1f}")

Is this good? Yes, it is very good! Let's compare with another solid hitter for contrast. Let's look at Nolan Arenado from the St. Louis Cardinals in 2025.

In [ ]:
# Find Nolan Arenado in 2025 data
arenado = batting_2025[batting_2025['Name'].str.contains('Nolan Arenado', case=False)].copy()

if len(arenado) > 0:
    arenado['X1B'] = arenado['H'] - arenado['2B'] - arenado['3B'] - arenado['HR']
    arenado['BB_HBP'] = arenado['BB'] + arenado['IBB'] + arenado['HBP']
    arenado['Outs'] = 0.982 * arenado['AB'] - arenado['H'] + arenado['GDP'] + arenado['SF'] + arenado['SH'] + arenado['CS']

    arenado_outs = arenado['Outs'].values[0]
    arenado_scale = (avg_outs - arenado_outs) / avg_outs

    arenado_stats = pd.DataFrame({
        'R': arenado['R'].values[0],
        'X1B': arenado['X1B'].values[0],
        'X2B': arenado['2B'].values[0],
        'X3B': arenado['3B'].values[0],
        'HR': arenado['HR'].values[0],
        'BB_HBP': arenado['BB_HBP'].values[0]
    }, index=[0])

    avg_team_scaled_arenado = avg_team[['R', 'X1B', 'X2B', 'X3B', 'HR', 'BB_HBP']] * arenado_scale
    avg_team_w_arenado = pd.DataFrame([avg_team_scaled_arenado.values + arenado_stats.values[0]],
                                   columns=['R', 'X1B', 'X2B', 'X3B', 'HR', 'BB_HBP'])

    runs_w_arenado = lm_runs.predict(avg_team_w_arenado)[0]
    raa_arenado = runs_w_arenado - avg_team_runs
    print(f"Nolan Arenado's Runs Above Average (2025): {raa_arenado:.1f}")
else:
    print("Nolan Arenado not found in qualified batters")

Comparing Judge to Arenado shows the difference between an elite power hitter and an above-average veteran player.

## Analyzing Defense -- Pitching

So far, all of our baseball analysis has looked at offensive production, commonly referred to as "hitting." Of course, this is only part of the game. Nearly half the players on a major league baseball team are pitchers, whose main job of course is to pitch the ball in such a way that the hitting team does not score runs. We now look at some analysis of pitching.

It can be difficult to isolate the effects of pitching. Traditional metrics like wins do not tell the whole story. For example, if a pitcher happens to pitch on days when the offense scores 9 runs, he can give up 7 runs over 5 innings and earn the "win" even though in reality he pitched rather poorly. Similarly, a defense behind the pitcher that makes many throwing and catching errors can make the pitcher look artificially bad.

*Fielding Independent Pitching* (FIP) is a statistic that, according to FanGraphs, is "what a player's 'Earned Run Average' would look like over a given period of time if the pitcher were to have experienced league average results on balls in play and league average timing." The formula for FIP is defined as:

$$
FIP = \frac{13 \cdot HR + 3 \cdot (BB + HBP) - 2 \cdot SO}{IP} + constant.
$$

The constant ensures that FIP is on the same scale as earned run average (ERA). The constant changes year-to-year but generally averages around 3.10.

In [ ]:
# Get team pitching data for multiple years
pitch_years = []
for year in range(1970, 2026):
    try:
        pitch_years.append(team_pitching_mlb(year))
    except:
        pass

df_team_pitch = pd.concat(pitch_years, ignore_index=True)

Now compare a team's calculated FIP to its ERA using a scatterplot.

In [ ]:
#| label: fig-baseball-fip-era
#| fig-cap: "Field independent pitching (x) versus earned run average (y) for teams since 1970."

sns.scatterplot(data=df_team_pitch, x='FIP_calc', y='ERA', alpha=0.25)
plt.xlabel('FIP (calculated)')
plt.ylabel('ERA')
plt.title('FIP vs ERA (Teams since 1970)')
plt.show()

We can see that team FIP and ERA are fairly well correlated, which should give us some faith in the formula used for FIP. However, there are outliers that should reflect teams with excellent (and poor) defenses relative to league averages.

## Statcast

### What is Statcast? {.unnumbered}

Statcast is Major League Baseball's ball and player tracking system that has been in every MLB ballpark since 2015. The current system has two components: (1) Trackman Doppler Radar tracks baseball events and (2) Hawk-Eye Cameras track player movements. In the first three seasons alone, 2.1 million pitches and 400,000 balls in play were tracked. In a nutshell, the location of the players and the ball is sampled multiple times per second to allow for a 3D re-creation of the game, but also can be used to evaluate optimal defensive placement of the players for particular batters or in certain game situations.

![Statcast](fig/statcast.png){#fig-statcast}

### pybaseball and Statcast {.unnumbered}

We will use the `pybaseball` package to query the MLB Baseball Savant (searchable Statcast) website. The pybaseball package can also be used to query FanGraphs, Baseball Reference, and other sites as well.

### Players and Player IDs {.unnumbered}

Suppose we want to look up players and query the Statcast API. Let's do this in order to look at data on the Los Angeles Dodgers' designated hitter (and pitcher) Shohei Ohtani. The first function that we will use is `playerid_lookup()`.

![Shohei Ohtani](fig/ohtani.jpeg){#fig-ohtani width=40%}

In [ ]:
# Look up Shohei Ohtani's player ID
ohtani_id = playerid_lookup('Ohtani', 'Shohei')
print(ohtani_id[['name_last', 'name_first', 'key_mlbam']])

![Babe Ruth](fig/ruth.jpeg){#fig-babe width=40%}

Shohei Ohtani, the holder of the record for the largest contract in MLB history, is a special case because he is both an everyday designated hitter *and* a starting pitcher. He is the first person to do this since the legendary Babe Ruth. The correct ID for querying Statcast is the one listed in the `key_mlbam` field. Now we can use that ID to get all of his information from any season of interest.

In [ ]:
# Get Ohtani's 2025 batting data from Statcast
ohtani_mlbam = ohtani_id['key_mlbam'].values[0]
df_ohtani_bat = statcast_batter('2025-03-27', '2025-09-28', player_id=ohtani_mlbam)
print(f"Total pitches to Ohtani in 2025: {len(df_ohtani_bat)}")

To compare, we will also look at defensive measures with Statcast. Pittsburgh Pirates' pitcher Paul Skenes burst onto the scene in 2024 and has quickly become one of the premier pitchers in baseball. Let's take a look at his 2025 pitching data.

In [ ]:
# Look up Paul Skenes and get his pitching data
skenes_id = playerid_lookup('Skenes', 'Paul')
skenes_mlbam = skenes_id['key_mlbam'].values[0]
df_skenes_pitch = statcast_pitcher('2025-03-27', '2025-09-28', player_id=skenes_mlbam)
print(f"Total pitches by Skenes in 2025: {len(df_skenes_pitch)}")

![Paul Skenes](fig/skenes.jpeg){#fig-skenes width=40%}

### Batting {.unnumbered}

With batters, we are sometimes interested in "advanced" metrics like `launch_speed`, `launch_angle`, and `hit_distance_sc`. These metrics provide different attributes about how hard, high, and far a ball is hit in the air by the player.

| **Abbreviation** | **Pitch Type** |
|------------------|----------------|
| CH | Change-up |
| CU | Curveball |
| FC | Cutter |
| FF | Four-Seam Fastball |
| FS | Splitter |
| KC | Knuckle Curve |
| SI | Sinker |
| SL | Slider |

: Pitch type abbreviations in Statcast. {#tbl-pitch-types}

Let's examine the type of pitches thrown to Shohei Ohtani in 2025.

In [ ]:
#| label: fig-baseball-ohtani-bat
#| fig-cap: "Ordered bar chart showing the proportion of pitch types thrown to Shohei Ohtani in 2025."

# Calculate proportion of each pitch type
pitch_counts = df_ohtani_bat['pitch_type'].value_counts(normalize=True).reset_index()
pitch_counts.columns = ['pitch_type', 'proportion']
pitch_counts = pitch_counts.sort_values('proportion', ascending=True)

# Plot
plt.figure(figsize=(8, 6))
plt.barh(pitch_counts['pitch_type'], pitch_counts['proportion'], color='gray')
plt.xlabel('Proportion of Pitches')
plt.ylabel('Pitch Type')
plt.title('Types of Pitches Thrown to Shohei Ohtani in 2025')
plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
plt.tight_layout()
plt.show()

This shows us that by far Shohei Ohtani has more four-seam fastballs (FF) thrown to him than any other type of pitch.

### Ohtani's Exit Velocity {.unnumbered}

We will use a violin plot to show the relative density of exit velocities for each pitch type.

In [ ]:
#| label: fig-baseball-ohtani-ev
#| fig-cap: "Violin plots showing Shohei Ohtani's exit velocity by pitch type thrown to him in 2025."

# Filter for common pitch types and balls in play
common_pitches = ['CH', 'CU', 'FC', 'FF', 'SI', 'SL', 'FS']
ohtani_in_play = df_ohtani_bat[
    (df_ohtani_bat['pitch_type'].isin(common_pitches)) &
    (df_ohtani_bat['type'] == 'X') &
    (df_ohtani_bat['launch_speed'].notna())
].copy()

plt.figure(figsize=(10, 6))
sns.violinplot(data=ohtani_in_play, x='pitch_type', y='launch_speed', color='gray')
plt.xlabel('Pitch Type')
plt.ylabel('Exit Velocity (mph)')
plt.title("Shohei Ohtani's Exit Velocity by Pitch Type in 2025")
plt.show()

### Visualizing Pitcher Statcast Data {.unnumbered}

Much like earlier where we analyzed the pitches that were thrown to Shohei Ohtani, we can now see the pitches that Paul Skenes threw to all batters.

In [ ]:
#| label: fig-baseball-skenes-bar
#| fig-cap: "Bar chart showing the proportion of pitch types thrown by Paul Skenes in 2025."

skenes_pitch_counts = df_skenes_pitch['pitch_type'].value_counts(normalize=True).reset_index()
skenes_pitch_counts.columns = ['pitch_type', 'proportion']
skenes_pitch_counts = skenes_pitch_counts.sort_values('proportion', ascending=True)

plt.figure(figsize=(8, 6))
plt.barh(skenes_pitch_counts['pitch_type'], skenes_pitch_counts['proportion'], color='gray')
plt.xlabel('Proportion of Pitches')
plt.ylabel('Pitch Type')
plt.title('Types of Pitches Thrown by Paul Skenes in 2025')
plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
plt.tight_layout()
plt.show()

### Skenes's Spin Rate {.unnumbered}

We can also visualize the spin rate for each of Skenes's pitch types.

In [ ]:
#| label: fig-baseball-skenes-spin
#| fig-cap: "Violin plots showing the spin rate by pitch types thrown by Paul Skenes in 2025."

skenes_common = ['FF', 'ST', 'FS', 'CH', 'SI']
skenes_in_play = df_skenes_pitch[
    (df_skenes_pitch['pitch_type'].isin(skenes_common)) &
    (df_skenes_pitch['release_spin_rate'].notna())
].copy()

plt.figure(figsize=(10, 6))
sns.violinplot(data=skenes_in_play, x='pitch_type', y='release_spin_rate', color='gray')
plt.xlabel('Pitch Type')
plt.ylabel('Spin Rate (rpm)')
plt.title("Paul Skenes's Spin Rate by Pitch Type in 2025")
plt.show()

## Strike Zones

When looking at strike zones, or the XYZ space in which a pitch is declared to be a strike, it will vary by batter (height) and to some extent even tendencies of the umpire. However, we can use an "average" zone as the basis for our figures.

### Creating the Strike Zone {.unnumbered}

A reasonable definition of a strike zone states the bottom of the strike zone (on average) as 1.5 feet above the ground and the top as 3.6 feet. Given that home plate is 17 inches wide, and "only a part of the ball need pass over home plate in order for the pitch to be called a strike", we define the strike zone width to be $17 + 2 \times (9/\pi)$ inches.

In [ ]:
plate_width = 17 + 2 * (9 / np.pi)  # in inches
plate_width_ft = plate_width / 12   # convert to feet

Now we can plot this zone as an outline on a graph where we will plot all of the pitches delivered to Shohei Ohtani.

In [ ]:
#| label: fig-baseball-strike-zone
#| fig-cap: "All pitch locations to Shohei Ohtani during the 2025 season."

from matplotlib.patches import Rectangle

fig, ax = plt.subplots(figsize=(8, 8))

# Plot all pitches
ax.scatter(df_ohtani_bat['plate_x'], df_ohtani_bat['plate_z'], alpha=0.2, s=10)

# Draw strike zone
strike_zone = Rectangle(
    (-plate_width_ft/2, 1.5),  # bottom-left corner
    plate_width_ft,             # width
    2.1,                        # height (3.6 - 1.5)
    fill=False,
    edgecolor='red',
    linewidth=2
)
ax.add_patch(strike_zone)

ax.set_xlabel('Horizontal Location (ft)')
ax.set_ylabel('Vertical Location (ft)')
ax.set_xlim(-4, 4)
ax.set_ylim(-1, 6)
ax.set_aspect('equal')
ax.set_title('Pitch Locations to Shohei Ohtani (2025)')
plt.show()

### Good Eye Shohei? {.unnumbered}

Next we can identify which pitches delivered to Ohtani caused him to swing, and for which ones he did not swing.

In [ ]:
#| label: fig-baseball-strike-zone-col
#| fig-cap: "All pitch locations to Shohei Ohtani during the 2025 season colored by whether he swung."

# Create swing indicator
no_swing_descriptions = ['ball', 'blocked_ball', 'automatic_ball', 'called_strike', 'hit_by_pitch']
df_ohtani_bat['swing'] = ~df_ohtani_bat['description'].isin(no_swing_descriptions)

fig, ax = plt.subplots(figsize=(8, 8))

# Plot swings and takes with different colors
colors = df_ohtani_bat['swing'].map({True: 'red', False: 'blue'})
ax.scatter(df_ohtani_bat['plate_x'], df_ohtani_bat['plate_z'],
           c=colors, alpha=0.3, s=10)

# Draw strike zone
strike_zone = Rectangle(
    (-plate_width_ft/2, 1.5), plate_width_ft, 2.1,
    fill=False, edgecolor='black', linewidth=2
)
ax.add_patch(strike_zone)

ax.set_xlabel('Horizontal Location (ft)')
ax.set_ylabel('Vertical Location (ft)')
ax.set_xlim(-4, 4)
ax.set_ylim(-1, 6)
ax.set_aspect('equal')
ax.set_title('Ohtani Swings (red) vs Takes (blue)')

# Add legend
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], marker='o', color='w', markerfacecolor='red', label='Swing'),
                   Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', label='Take')]
ax.legend(handles=legend_elements)
plt.show()

## Estimating Probabilities

While both authors are big fans of Shohei Ohtani, there are other players in the league. Let's examine the data from another star player, Aaron Judge.

Managers of baseball teams have to make many in-game decisions to implement their strategy and have the best chance of success. Probabilities are what drive these decisions.

In [ ]:
# Get Aaron Judge's 2025 Statcast data
judge_lookup = playerid_lookup('Judge', 'Aaron')
judge_mlbam = judge_lookup['key_mlbam'].values[0]
df_judge_bat = statcast_batter('2025-03-27', '2025-09-28', player_id=judge_mlbam)
print(f"Total pitches to Judge in 2025: {len(df_judge_bat)}")

### Launch Angle {.unnumbered}

Using the above data set, we can create a density plot showing the launch angle for all balls Judge put into play in 2025.

In [ ]:
#| label: fig-baseball-judge-den
#| fig-cap: "Density plot for launch angle for Aaron Judge's balls that were put into play during the 2025 season."

judge_in_play = df_judge_bat[df_judge_bat['type'] == 'X'].copy()

sns.kdeplot(data=judge_in_play, x='launch_angle')
plt.xlabel('Launch Angle')
plt.ylabel('Density')
plt.title("Aaron Judge's Launch Angle Distribution (2025)")
plt.show()

Suppose we want to know the probability of a hit given the launch angle? Let's first define what counts as a hit.

In [ ]:
# Define hits
hit_events = ['single', 'double', 'triple', 'home_run']
df_judge_hits = judge_in_play[judge_in_play['launch_angle'].notna()].copy()
df_judge_hits['hit'] = df_judge_hits['events'].isin(hit_events).astype(int)

print(f"Total balls in play: {len(df_judge_hits)}")
print(f"Total hits: {df_judge_hits['hit'].sum()}")

We will then create a plot with points showing hit or not a hit on the y-axis and launch angle on the x axis, with a smooth curve.

In [ ]:
#| label: fig-baseball-judge-la-hit
#| fig-cap: "Aaron Judge's hits (1) or not (0) along with their launch angle during the 2025 season."

plt.figure(figsize=(10, 6))
plt.scatter(df_judge_hits['launch_angle'], df_judge_hits['hit'], alpha=0.15)

# Add lowess smooth
from statsmodels.nonparametric.smoothers_lowess import lowess
smooth = lowess(df_judge_hits['hit'], df_judge_hits['launch_angle'], frac=0.3)
plt.plot(smooth[:, 0], smooth[:, 1], color='black', linewidth=2)

plt.xlabel('Launch Angle')
plt.ylabel('Hit (1) or Out (0)')
plt.title("Aaron Judge's Hit Probability by Launch Angle (2025)")
plt.xticks(range(-60, 100, 20))
plt.show()

This tells us that his highest probability of getting a hit is with a launch angle of about 15-25 degrees.

## Logistic Regression Model

One method for modeling the probability of an event with two outcomes is to use logistic regression. We use logistic regression more as a classification model, as we are trying to classify an output as one of two possibilities (in this case, a hit or not a hit).

In [ ]:
# Fit logistic regression model
glm_judge = smf.logit('hit ~ launch_angle', data=df_judge_hits).fit(disp=0)
print(glm_judge.summary())

Note that if the coefficient related to a predictor variable is greater than zero, then the probability of `Y=1` increases as the predictor variable increases.

In [ ]:
#| label: fig-baseball-judge-prob-la-hit
#| fig-cap: "Estimated probability of Aaron Judge getting a hit as a function of launch angle."

# Create prediction data
la_range = np.linspace(df_judge_hits['launch_angle'].min(),
                        df_judge_hits['launch_angle'].max(), 100)
pred_data = pd.DataFrame({'launch_angle': la_range})
pred_data['prob'] = glm_judge.predict(pred_data)

plt.figure(figsize=(10, 6))
plt.scatter(df_judge_hits['launch_angle'], df_judge_hits['hit'], alpha=0.15)
plt.plot(pred_data['launch_angle'], pred_data['prob'], color='black', linewidth=2)
plt.xlabel('Launch Angle')
plt.ylabel('P(Hit)')
plt.title("Logistic Regression: Hit Probability by Launch Angle")
plt.show()

This simple model shows a fairly constant decrease as launch angle increases. But we know probability should increase and then decrease. Let's add a quadratic term.

In [ ]:
# Fit logistic regression with quadratic term
df_judge_hits['launch_angle_sq'] = df_judge_hits['launch_angle'] ** 2
glm_judge_2 = smf.logit('hit ~ launch_angle + launch_angle_sq', data=df_judge_hits).fit(disp=0)
print(glm_judge_2.summary())

In [ ]:
#| label: fig-baseball-judge-prob-la-hit-quad
#| fig-cap: "Estimated probability of Aaron Judge getting a hit using a quadratic logistic regression model."

pred_data['launch_angle_sq'] = pred_data['launch_angle'] ** 2
pred_data['prob_quad'] = glm_judge_2.predict(pred_data)

plt.figure(figsize=(10, 6))
plt.scatter(df_judge_hits['launch_angle'], df_judge_hits['hit'], alpha=0.15)
plt.plot(pred_data['launch_angle'], pred_data['prob_quad'], color='black', linewidth=2)
plt.xlabel('Launch Angle')
plt.ylabel('P(Hit)')
plt.title("Quadratic Logistic Regression: Hit Probability by Launch Angle")
plt.show()

This quadratic model makes more sense - it shows that there's an optimal launch angle around 15-20 degrees where hit probability is maximized.

## Summary

In the century and a half that professional baseball has been played in the United States, there have been tons of data generated. Yet this will only continue to grow as much more data is collected every day. Camera systems in major league ballparks now record the positions of every player and the ball thirty times per second. We have only begun to scratch the surface here of what we can do with data and baseball.

Fortunately for those of us who do not work for major league baseball teams, the volume of this data that is available to the public for free allows us to do our own analysis. We used the pybaseball package to gather information about both teams and players, and we used it to not only see what has happened in the past, but also to predict the future in terms of how many runs will a team score and then how many games will they win. We hope this chapter not only introduces you to baseball analytics and improves your Python programming skills, but also that it inspires you to create your own analysis!

___

## Lab Four: Calculating Baseball Metrics

Earlier in the chapter we went through the details of calculating Runs Created. Now we can go through a similar methodology to calculate some of the other metrics we will use in baseball analysis.

1. **OBP, SLG, and OPS**: Add On-Base Percentage (OBP), Slugging Percentage (SLG), and On Base Plus Slugging (OPS) to your team data.

$$OBP = \frac{H + BB + HBP}{AB + BB + HBP + SF}$$

$$SLG = \frac{1B + 2 \cdot 2B + 3 \cdot 3B + 4 \cdot HR}{AB}$$

$$OPS = OBP + SLG$$

In [ ]:
#| eval: false
#| include: false

rc_df['OBP'] = (rc_df['H'] + rc_df['BB'] + rc_df['HBP']) / (rc_df['AB'] + rc_df['BB'] + rc_df['HBP'])
rc_df['SLG'] = (rc_df['X1B'] + 2*rc_df['2B'] + 3*rc_df['3B'] + 4*rc_df['HR']) / rc_df['AB']
rc_df['OPS'] = rc_df['OBP'] + rc_df['SLG']

2. Compute a correlation matrix for `R`, `RC`, `AB`, `H`, `HR`, `OBP`, `SLG`, and `OPS` and plot the results.

3. **Base Runs (BsR)**: An alternative to `RC` created by David Smythe. The formula is:
$$BsR = \frac{A \cdot B}{B + C} + D$$
where $A = H + BB - HR$, $B = (1.4 \cdot TB - 0.6 \cdot H - 3 \cdot HR + 0.1 \cdot BB) \cdot 1.02$, $C = AB - H$, and $D = HR$.

4. Compare simple linear regressions of `R` on each of `RC` and `BsR`. Which gives the best fit?

5. Using the `batting_stats()` function from pybaseball, fit a regression model to predict runs based on singles, doubles, triples, home runs, and walks + HBP.

___

## Case Study: Adjusting to Baseball in Colorado

When Major League Baseball expanded to Colorado in 1993, it was a live experiment in how the game would change. It was long known that objects tend to travel farther for a given level of force when the altitude level increases. Before the league expansion in 1993, the MLB ballpark with the highest altitude was Atlanta's Fulton County Stadium at approximately 1100 feet. How would baseball change with a ballpark in Denver, the "Mile High City" which is almost five times higher than Atlanta?

![Coors Field and the famous row of purple seats at one mile above sea level.](fig/coors.jpeg){#fig-coors}

As predicted, in its first few years, home runs were hit at prodigious levels in Colorado. Coors Field was given the nickname "Coors Canaveral" and the players were known as the Blake Street Bombers.

In 2002, Coors Field installed a humidor where the baseballs would be kept prior to games, making the baseballs slightly heavier and easier for the pitchers to grip. This helped reduce the number of home runs hit in the stadium.

Despite all of these improvements, pitchers still complain that the baseball spins less from the pitcher's hand to home plate in Denver. This means that balls which may normally curve sharply at sea level will not curve as much at Coors Field.

Answer the following using data from the hitters or the pitchers:

1. How would you measure the effect of spin rate? Does it analytically differ in Coors Field from other parks?
2. If you were the manager, general manager, or pitching coach of the Colorado Rockies, how would you propose addressing the issues?
3. Based on your answers to the question above, how would you measure success?

___

## Exercises

1. Choose two teams from the pybaseball data. Define a new dataframe by filtering to include only those two teams. How many rows are in the new dataframe?

2. Using the `team_batting()` function, calculate Runs Created for all teams in the most recent season. Which team had the highest Runs Created value?

3. Fit a linear regression model predicting runs from home runs for players with at least 100 at-bats. What is the R-squared value?

4. Get Statcast data for a player of your choice. Create a visualization of their pitch locations colored by pitch type.

5. Fit a logistic regression model predicting hits based on exit velocity for any player. What exit velocity gives the highest predicted probability of a hit?